### acceso a openlibrary (no sirve)

In [ ]:
# ======================================================================================
# ACCESO A API DE OPENLIBRARY
# ======================================================================================

import requests
import json
import pandas as pd
from pathlib import Path
import re


# ======================================================================================
# LECTURA DE DATOS
# ======================================================================================

# --- Contribuidores (editor, traductor...)
def leer_brief(isbn):   

    # Llamada para contribuidores
    url = f"http://openlibrary.org/api/volumes/brief/isbn/{isbn}"

    response = requests.get(url)

    if response.status_code == 200:
        api1 = response.json()

        if api1 == []:
            # No encuentra el libro
            return {}
        
        contenido = api1["records"]

        # clave tipo /books/OL9130631M
        clave = list(contenido.keys())[0]
        
        # Edición exacta
        edicion = contenido[clave]["details"]["details"]["edition_name"]

        # Editores
        contribuidores = contenido[clave]["details"]["details"]["contributors"]
        editores = []
        if contribuidores != "":
            for contr in contribuidores:
                if contr['role'] == 'Editor':
                    editores.append(contr["name"])

        # Peso
        peso = contenido[clave]["details"]["details"]['weight']

        # Dimensiones
        dim = contenido[clave]["details"]["details"]['physical_dimensions']

        # Formato
        formato = contenido[clave]["details"]["details"]['physical_format']

        # Categorías
        total_cats = contenido[clave]["data"]["subjects"]
        categorias = []
        for cat_dict in total_cats:
            categorias.append(cat_dict["name"])
        categorias += contenido[clave]["details"]["details"]['subjects']
        # categorias.apply(lambda x: x.translate(str.maketrans({"-":"", "/": "", "&": "and"})).strip())
        

    else: 
        raise Exception(f"Conexión denegada. Status {response.status_code}")
    
    return {'edicion': edicion, 'editores': editores, 'peso': peso, 'dim': dim, 'formato': formato, 'subcategorias': categorias}

# --- Ratings
def leer_ratings(isbn):

    # Llamada para ratings
    url = f"https://openlibrary.org/search.json?isbn={isbn}&fields=rating*"

    response = requests.get(url)

    if response.status_code == 200:
        api2 = response.json()
        ratings = api2["docs"]

    else: 
        raise Exception(f"Conexión denegada. Status {response.status_code}")
    
    return ratings[0] if ratings else {}

# --- Sinopsis (ver como conseguirla de otro sitio)


# ======================================================================================
# CONTROL DE FLUJO INTERACTIVO
# ======================================================================================

def control_flujo_api(ruta_catalogos="data/prueba"): # cambiarlo para que te permita elegir cuántos libros hay que 

    print("Iniciando búsqueda en API...")
    catalogos = [f for f in Path(ruta_catalogos).iterdir() if f.is_file()]

    for ruta_cat in catalogos:
        print(f"\nLeyendo {ruta_cat.name}...")

        with open(ruta_cat, "r", encoding="utf-8") as f:
            catalogo = json.load(f)

        if 'openl' in catalogo[0].keys():
            print("Catálogo completo.")
            continue 
        
        # key = input("Continuar [Y/N]?")    

        # if key=='N':
        #     continue

        print(f"Comenzando con {ruta_cat.name}. Total de libros a buscar: {len(catalogo)+1}...")
        for libro in range(len(catalogo)):
            print(f"[{libro+1}/{len(catalogo)+1}]")
            ean = catalogo[libro]['EAN']

            brief = leer_brief(ean)
            ratings = leer_ratings(ean)

            if isinstance(brief, dict) and isinstance(ratings, dict):
                catalogo[libro].update({
                    **brief,
                    **ratings,
                    'portada': f"https://covers.openlibrary.org/b/isbn/{ean}-L.jpg",
                    'openl': True
                })
            else:
                catalogo[libro]['openl'] = False
                
            print("Catálogo terminado.")

        with open(ruta_cat, "w", encoding="utf-8") as f:
                    json.dump(catalogo, f, ensure_ascii=False, indent=2)


# ======================================================================================
# PUNTO DE ENTRADA
# ======================================================================================

control_flujo_api()


## Capa silver nueva versión

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
from src.constants import TRADUCTOR_EDITOR, OTROS_CONTRIBUIDORES, ILUSTRACIONES, ESCOLARES, CATEGORIAS, COLUMNAS_FINALES, CATEGORIAS, SUBCATEGORIAS, ENCUADERNACION, EDITORIALES, CATEGORIAS_SPI

# ======================================================================================
# CREACIÓN DEL DATAFRAME BASE
# ======================================================================================

def crear_df(ruta_catalogos="data/bronze/catalogos"):
    path = Path(ruta_catalogos)
    df = pd.DataFrame({})
    jsons = []

    print("="*50,"\nCreando DataFrame con todos los libros\n","="*50)
    for archivo in path.iterdir():
        if archivo.is_file():
            print(f"Añadiendo {archivo.name}")
            editorial = pd.read_json(archivo.absolute())
            jsons.append(editorial)

    df = pd.concat(jsons, axis=0)

    # Borrar filas repetidas
    print("Catálogos convertidos a DataFrame. Eliminando filas duplicadas...")
    df.drop_duplicates(subset=['EAN'], keep='first', inplace=True)

    # Limpiado de nombre de columnas
    df.columns = df.columns.str.strip().str.lower().str.translate(str.maketrans({"á": "a", "é": "e", "í": "i", "ó":"o", "ú": "u", "º": "", " ": "_"}))

    print("DataFrame creado con éxito.")
    return df


# =============================================================================
# FUNCIONES AUXILIARES Y LIMPIEZA BÁSICA
# =============================================================================

# Búsqueda de la moda en una lista de strings
def moda(x):
    """
    Devuelve la moda de una serie.
    """

    x = x.dropna()

    if len(x) == 0:
        return np.nan

    return x.mode().iloc[0]


# Extrae el número (precio, medida...) de un string
def extraer_numero(x):

    _NUMERO = re.compile(r"(\d+[.,]?\d*)")

    if pd.isna(x):
        return np.nan

    m = _NUMERO.search(str(x))

    if m is None:
        return np.nan

    return float(m.group(1).replace(",", "."))


# Conversión de los elementos de una lista/columna en listas
def normalizar_lista(valor):
    """
    Convierte cualquier valor en una lista.

    NaN -> []
    str -> [str]
    list -> list limpia
    ndarray -> list
    """

    if valor is None:
        return []

    if isinstance(valor, float) and np.isnan(valor):
        return []

    if isinstance(valor, str):
        valor = valor.strip().title()
        if valor == "":
            return []

        return [valor]

    if isinstance(valor, np.ndarray):
        valor = valor.tolist()

    if isinstance(valor, (list, tuple)):
        salida = []
        for x in valor:
            if pd.isna(x):
                continue

            x = str(x).strip().title()

            if x:
                salida.append(x)

        return list(dict.fromkeys(salida))

    return [str(valor)]


def normalizar_columnas_lista(df, columnas_listas):

    df = df.copy()

    for col in columnas_listas:
        if col in df.columns:
            df[col] = df[col].apply(normalizar_lista)

    return df


# Limpieza básica del DF (eliminar filas son datos obligatorios, duplicados, relleno de columnas nulas y mapeo)
def limpieza_basica(df,dict_editoriales=None,dict_encuadernacion=ENCUADERNACION):

    # quitar duplicados por EAN
    df.drop_duplicates(subset="ean", inplace=True)

    # eliminar libros sin autor y sin categoría
    df = df.dropna(
        subset=["ean", "titulo", "autoria", "categorias"],
        how="any",
    )
    df['ean'] = df['ean'].astype(str)

    # fecha
    df["fecha_publicacion"] = pd.to_datetime(
        df["fecha_publicacion"],
        format="%d-%m-%Y",
        errors="coerce",
    )

    # sinopsis
    df["sinopsis"] = df["sinopsis"].fillna("Sin sinopsis")

    # ids editoriales
    if dict_editoriales is not None:
        df["editorial"] = df["editorial"].map(dict_editoriales)

    # encuadernación
    if dict_encuadernacion is not None:
        df["encuadernacion"] = df["encuadernacion"].map(dict_encuadernacion)

    return df

def normalizar_titulos(nombre:str):
    articulos = {"El", "La", "Los", "Las", "Un", "Una", "Unos", "Unas"}

    if "," in nombre:
        titulo, articulo = map(str.strip, nombre.rsplit(",", 1))
        if articulo in articulos:
            nombre = f"{articulo} {titulo}"

    nombre = nombre.strip().title()

    return nombre 

# =============================================================================
# MERGE DE COLUMNAS Y FEATURES
# =============================================================================

# Unión de columnas para crear otra nueva
def merge_columnas(df, nombre, columnas):

    df = df.copy()

    for col in columnas:
        if col not in df.columns:
            df[col] = [[] for _ in range(len(df))]

    df[nombre] = df[columnas].sum(axis=1).apply(lambda x: list(dict.fromkeys(x)))

    return df

# Coversión de columnas numéricas que aparecen como str
def extraer_numeros(df):

    df = df.copy()

    for col in ["precio","peso","grueso","n_paginas"]:
        if col in df.columns:
            df[col] = df[col].apply(extraer_numero)

    # dimensiones (ej: 240 x 170 mm)
    medidas = df["dimensiones"].astype(str).str.extract(r"(\d+[.,]?\d*)\D+(\d+[.,]?\d*)")

    df["alto_mm"] = medidas[0].str.replace(",", ".", regex=False).astype(float)
    df["ancho_mm"] = medidas[1].str.replace(",", ".", regex=False).astype(float)

    return df

def crear_marcadores(df, ilustraciones, escolares):
    # Escolares
    escolar = (
        df[escolares]
        .apply(lambda col: col.str.len())
        .sum(axis=1)
        > 0
    )
    df['es_escolar'] = escolar 

    # Ilustrada
    ilustrada = (
        df[ilustraciones]
        .apply(lambda col: col.str.len())
        .sum(axis=1)
        > 0
    )
    df['es_ilustrada'] = ilustrada

    # Impresión bajo demanda
    ibd = (
        df["ibd"]
        .str.len()
        > 0
    )
    df['es_ibd'] = ibd

    return df


def crear_portada(df:pd.DataFrame):
    # URL imagen
    ean = df["ean"].astype(str)
    df["img"] = (
        "https://static.cegal.es/imagenes/marcadas/"
        + ean.str[:8]
        + "/"
        + ean
        + ".gif"
    )

    return df


def definir_aparato_critico(df, cols_ap):

    def tiene_contenido(valor):
        if valor is None:
            return False
        if isinstance(valor, float) and pd.isna(valor):
            return False
        if isinstance(valor, (list, np.ndarray)) and len(valor) == 0:
            return False
        return True

    # Creamos la máscara booleana
    mask = df[cols_ap].map(tiene_contenido).any(axis=1)
    df["aparato_critico"] = mask

    # Generamos los valores directamente en el apply sin necesidad de .loc
    def extraer_tipos(fila):
        presentes = [col for col in cols_ap if tiene_contenido(fila[col])]
        return presentes if presentes else np.nan

    df["tipo_aparato_critico"] = df[cols_ap].apply(extraer_tipos, axis=1)

    return df

def rellenar_columnas(df):

    df = df.copy()

    # Medidas por editorial + colección
    for col in ["alto_mm", "ancho_mm", "precio", "n_paginas"]:
        mediana_col = df.groupby(["editorial", "coleccion"])[col].transform("median")
        mediana_enc = df.groupby(["editorial", "encuadernacion"])[col].transform("median")
        
        df[col] = df[col].fillna(mediana_col).fillna(mediana_enc)


    # Grosor (fórmula estándar)
    df["grueso"] = df["grueso"].fillna(df["n_paginas"] * 0.04)

    # Peso (fórmula estándar)
    peso_estimado = df['peso'].fillna((df["alto_mm"]/1000) * (df["ancho_mm"]/1000) * (df["n_paginas"]/2) * 80 + 120)

    df["peso"] = df["peso"].fillna(peso_estimado)

    # Idioma original 
    df["autor_principal"] = (
        df["autoria"]
        .apply(
            lambda x:
                x[0]
                if len(x)
                else np.nan
        )
    )

    idioma = (
        df.groupby("autor_principal")[
            "idioma_original"
        ]
        .transform(moda)
    )

    df["idioma_original"] = df["idioma_original"].fillna(idioma)

    df.drop(columns="autor_principal", inplace=True)

    return df


# inferencia categorías
def inferencia_categoria(df, categorias=SUBCATEGORIAS):

    def obtener_categorias(subcategorias_libro):
        if not isinstance(subcategorias_libro, (list, tuple, set)):
            return []

        return list({
            categoria
            for subcategoria in subcategorias_libro
            for categoria, subcategorias in categorias.items()
            if subcategoria in subcategorias
        })

    df["categorias"] = df["subcategorias"].apply(obtener_categorias)

    return df


# =============================================================================
# LIMPIEZA FINAL
# =============================================================================

def limpiar_columnas(df, cols_borrar):

    borrar = [c for c in cols_borrar if c in df.columns]

    return df.drop(columns=borrar)


# =============================================================================
# PIPELINE
# =============================================================================

def limpiar_df_completa(data, dict_editoriales=EDITORIALES, dict_encuadernacion=ENCUADERNACION):

    df = data.copy()
    TTL_A_ED = {
        nombre_ttl: editorial
        for editorial, datos in dict_editoriales.items()
        for nombre_ttl in datos["ttl"]
    }

    # Cambio de nombres de columnas
    columnas_lista = list(set(
        TRADUCTOR_EDITOR
        + OTROS_CONTRIBUIDORES
        + ILUSTRACIONES
        + ESCOLARES
        + CATEGORIAS
        + ["autoria"]
    ))
    df = normalizar_columnas_lista(df, columnas_lista)

    # Limpieza
    df = limpieza_basica(df, TTL_A_ED, dict_encuadernacion)
    df['titulo'] = df['titulo'].apply(normalizar_titulos)

    # Merge colaboradores
    df = merge_columnas(df, "traductor_y_editor", TRADUCTOR_EDITOR)
    df = merge_columnas(df,"otros_contribuidores", OTROS_CONTRIBUIDORES)
    df = merge_columnas(df, "subcategorias", CATEGORIAS)

    # Feature engineering
    df = extraer_numeros(df)
    df = definir_aparato_critico(df, OTROS_CONTRIBUIDORES)
    df = crear_marcadores(df, ESCOLARES, ILUSTRACIONES)
    df = crear_portada(df)

    # Relleno
    df = rellenar_columnas(df)
    df = inferencia_categoria(df)

    # Limpieza final
    borrar = (
        TRADUCTOR_EDITOR
        + OTROS_CONTRIBUIDORES
        + ILUSTRACIONES
        + ['isbn']
    )
    df = limpiar_columnas(df, borrar)

    df = df[[c for c in COLUMNAS_FINALES if c in df.columns]]

    return df


def validar_catalogo(df):
    # EAN únicos
    if df['ean'].nunique().count() < len(df['ean']):
        print('Existen números EAN repetidos.') 
    
    # sin nulos en columnas obligatorial
    cols_nulos = ["ean", "titulo", "autoria", "categorias"]
    for col in cols_nulos:
        if df[col].isna().sum() > 0:
            print(f'Existen nulos en la columna {col}.') 

    # fechas válidas
    formato = '%d/%m/%Y'
    for fecha in df['fecha_publicacion']:
        try:
            fecha_valida = datetime.strptime(fecha, formato)
            print("Fecha correcta")
        except ValueError:
            print("Fecha inválida")

    # dimensiones positivas
    cols_numericas = ["n_paginas","precio","alto_mm","ancho_mm","grueso","peso"]
    for col in cols_numericas:
        if any(x<=0 for x in df[col]):
            print(f"La columna {col} tiene núemeros no positivos.")


In [110]:
import json
import pandas as pd 
import numpy as np 
from pathlib import Path
from src.constants import EDITORIALES
SPI_A_ED = {
    nombre_spi: editorial
    for editorial, datos in EDITORIALES.items()
    for nombre_spi in (
        datos["spi"]
        if isinstance(datos["spi"], list)
        else [datos["spi"]]
    )
    if nombre_spi is not None
}

def merge_spi(ruta_spi="data/bronze/spi", spi_a_ed = SPI_A_ED):
    rutas = sorted(Path(ruta_spi).glob("*.csv"))
    dfs = []
    for ruta in rutas:
        df = pd.read_csv(ruta)

        if "Editorial" not in df.columns:
            raise ValueError(f"{ruta.name} no contiene la columna 'Editorial'.")

        nombre = ruta.stem.lower().replace("clasificacion_", "")

        df = df.rename(columns={
            c: f"{c}_{nombre}"
            for c in df.columns
            if c != "Editorial"
        })
        df["Editorial"] = df["Editorial"].map(spi_a_ed)
        df = (
            df
            .groupby("Editorial", as_index=False)
            .first()
        )
        df = df.set_index("Editorial")
        dfs.append(df)
    
    df = pd.concat(dfs, axis=1, join="outer").reset_index()

    df_selection = df[df['Editorial'].isin(spi_a_ed.values())].copy()
    print(df_selection['Editorial'])
    return df_selection

def prestigio_editorial(df):
    df = df.loc[:, ~df.columns.duplicated()].copy()

    df_prestigio = pd.DataFrame({})
    df_prestigio["editorial"] = df["Editorial"]

    for i in range(1, len(df.columns) - 1, 2):
        col_pos = df.columns[i]
        col_icee = df.columns[i + 1]

        nombre_col = f"prestigio{str(col_pos).replace('Posición', '').strip()}"

        # Extraer como Series unimodales e iloc para evitar ambigüedades
        s_pos = df.iloc[:, i]
        s_icee = df.iloc[:, i + 1]

        mask = s_pos.notna() & s_icee.notna()

        icee = s_icee
        icee_min = icee.min()
        icee_max = icee.max()
        
        if icee_max != icee_min:
            icee_norm = (icee - icee_min) / (icee_max - icee_min)
        else:
            icee_norm = pd.Series(0.0, index=df.index)

        n = s_pos.max()
        percentil = 1 - (s_pos - 1) / (n - 1) if (pd.notna(n) and n > 1) else pd.Series(1.0, index=df.index)

        df_prestigio[nombre_col] = 0.0
        
        # Asignación segura con valores indexados por la máscara
        val_calculado = 0.1 + 0.9 * (0.8 * icee_norm[mask] + 0.2 * percentil[mask])
        df_prestigio.loc[mask, nombre_col] = val_calculado

    df_prestigio = df_prestigio.fillna(0.0)

    # Se incluyen las editoriales que no están en el SPI
    nuevas_filas = {}
    for col in df_prestigio.columns:
        nuevas_filas[col] = []

    for ed_dict in EDITORIALES.items():
        ed = ed_dict[0]
        spi = ed_dict[1]['spi']

        if spi is None:
            for col in nuevas_filas.keys():
                if col == 'editorial':
                    nuevas_filas[col].append(ed)
                else: 
                    nuevas_filas[col].append(0)

    nuevas_filas = pd.DataFrame(nuevas_filas)
    df_prestigio = pd.concat([df_prestigio, nuevas_filas], ignore_index=True)

        # df_prestig.to_parquet("data/silver/prestigio_spi.parquet", engine="pyarrow")

        # añadir filas de editoriales que no están en el spi con todo cero

    return df_prestigio

In [111]:
merge_spi()

0      Alianza Editorial
1         Ediciones Akal
2      Ediciones Cátedra
3                Destino
4             Acantilado
5               Anagrama
6                Siruela
7                 Gredos
8              Alfaguara
9     Castalia Ediciones
10                Espasa
11                   RAE
Name: Editorial, dtype: str


,Editorial,Posición_antropologia,ICEE_antropologia,Posición_arqueologia,ICEE_arqueologia,Posición_bellas_artes,ICEE_bellas_artes,Posición_biblioteconomia,ICEE_biblioteconomia,Posición_comunicacion,...,Posición_historia,ICEE_historia,Posición_literatura,ICEE_literatura,Posición_politica,ICEE_politica,Posición_psicologia,ICEE_psicologia,Posición_sociologia,ICEE_sociologia
0,Alianza Editorial,16.0,2.0,NaN,NaN,6.0,32.0,NaN,NaN,5.0,...,4.0,105.0,10.0,59.0,1.0,17.0,4.0,53.0,1.0,90.0
1,Ediciones Akal,15.0,3.0,2.0,30.0,1.0,87.0,NaN,NaN,20.0,...,3.0,109.0,6.0,89.0,NaN,NaN,NaN,NaN,10.0,20.0
2,Ediciones Cátedra,6.0,12.0,4.0,17.0,3.0,44.0,10.0,5.0,1.0,...,9.0,56.0,1.0,210.0,NaN,NaN,NaN,NaN,18.0,8.0
3,Destino,NaN,NaN,9.0,6.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Acantilado,NaN,NaN,NaN,NaN,22.0,6.0,NaN,NaN,23.0,...,33.0,8.0,40.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
5,Anagrama,NaN,NaN,NaN,NaN,13.0,18.0,NaN,NaN,13.0,...,NaN,NaN,23.0,22.0,NaN,NaN,NaN,NaN,NaN,NaN
6,Siruela,NaN,NaN,NaN,NaN,11.0,20.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Gredos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,34.0,7.0,3.0,155.0,NaN,NaN,NaN,NaN,NaN,NaN
8,Alfaguara,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,28.0,15.0,NaN,NaN,NaN,NaN,NaN,NaN
9,Castalia Ediciones,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,18.0,31.0,NaN,NaN,NaN,NaN,NaN,NaN


## Capa gold

In [85]:
import pandas as pd
import numpy as np
import ast
from sklearn.preprocessing import MinMaxScaler
from src.constants import SUBCATEGORIAS

def merge_dataframes(ttl: pd.DataFrame, spi: pd.DataFrame):
    return ttl.merge(spi, how="left", on="editorial", validate="many_to_one")

def portabilidad(df):
    columnas = ["alto_mm", "ancho_mm", "grueso", "peso"]
    scaler = MinMaxScaler()

    escaladas = pd.DataFrame(
        scaler.fit_transform(df[columnas]),
        columns=columnas,
        index=df.index
    )

    # Invertir: 1 = pequeño/ligero = más portátil
    # .clip(lower=0.01) evita que un valor máximo ponga a 0 toda la media geométrica
    escaladas = (1 - escaladas).clip(lower=0.01)

    df["indice_portabilidad"] = escaladas.prod(axis=1) ** (1 / len(columnas))
    return df

def compacidad(df):
    df['indice_compacidad'] = df['n_paginas'] / (df['alto_mm'] * df['ancho_mm'] * df['grueso'])
    return df

def prestancia(df):
    variables = ["alto_mm", "ancho_mm", "grueso", "peso"]
    scaler = MinMaxScaler()

    normalizadas = pd.DataFrame(
        scaler.fit_transform(df[variables]),
        columns=variables,
        index=df.index
    )

    pesos_encuadernacion = {
        "rústica": 0.4,
        "rústica con solapas": 0.5,
        "tapa blanda": 0.4,
        "cartoné": 0.7,
        "tapa dura": 0.8,
        "tela": 0.9,
        "piel": 1.0,
    }

    encuadernacion = (
        df["encuadernacion"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
        .map(pesos_encuadernacion)
        .fillna(0.5)
    )

    df["indice_prestancia"] = (
        0.20 * (normalizadas["alto_mm"] + normalizadas["ancho_mm"]) / 2
        + 0.15 * normalizadas["grueso"]
        + 0.20 * normalizadas["peso"]
        + 0.45 * encuadernacion
    )
    return df

def ap_critico(df, pesos=None):
    # Claves normalizadas SIN tildes para coincidir con tipo_aparato_critico
    if pesos is None:
        pesos = {
            "introduccion": 1.0,
            "prologo": 1.0,
            "epilogo": 1.0,
            "notas": 1.5,
            "anotaciones": 1.5,
            "estudio": 2.0,
            "comentarios": 1.5,
            "bibliografia": 1.0,
            "cronologia": 0.5,
            "trabajo_preliminar": 1.0
        }

    def calcular_score(textos):
        if isinstance(textos, str):
            try:
                textos = ast.literal_eval(textos)
            except (ValueError, SyntaxError):
                textos = []

        if not isinstance(textos, (list, tuple, set)):
            return 0.0

        return sum(pesos.get(str(texto).lower().strip(), 0) for texto in textos)

    df["score_critico"] = df["tipo_aparato_critico"].apply(calcular_score)
    return df

def colaboradores(df):
    def parse_lista(val):
        if isinstance(val, str):
            try:
                return ast.literal_eval(val)
            except (ValueError, SyntaxError):
                return []
        return val if isinstance(val, list) else []

    colabs_series = df["otros_contribuidores"].apply(parse_lista)
    frecuencia = colabs_series.explode().dropna().value_counts()

    def calcular_score(lista):
        if not isinstance(lista, (list, tuple, set)) or len(lista) == 0:
            return 0.0

        valores = [frecuencia.get(colaborador, 0) for colaborador in lista]
        if not valores:
            return 0.0

        return np.mean(np.log1p(valores))

    df["score_colaboradores"] = colabs_series.apply(calcular_score)
    return df

def prestigio(df):
    def parse_lista(val):
        if isinstance(val, str):
            try:
                return ast.literal_eval(val)
            except (ValueError, SyntaxError):
                return []
        return val if isinstance(val, list) else []

    def calcular_prestigio(fila):
        subcategorias = parse_lista(fila["subcategorias"])
        categorias = parse_lista(fila["categorias"])

        if not subcategorias or not categorias:
            return np.nan

        conteo = {}
        for subcategoria in subcategorias:
            for categoria in categorias:
                if categoria in SUBCATEGORIAS and subcategoria in SUBCATEGORIAS[categoria]:
                    conteo[categoria] = conteo.get(categoria, 0) + 1
                    break

        if not conteo:
            return np.nan

        total = sum(conteo.values())
        pesos = {cat: cant / total for cat, cant in conteo.items()}

        valores = []
        for categoria, peso in pesos.items():
            # Formato exacto de la columna en SPI/Gold
            columna = f"prestigio_{categoria.lower().replace(' ', '_').replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')}"

            if columna not in df.columns:
                continue

            valor = fila[columna]
            if pd.notna(valor):
                valores.append((valor, peso))

        if not valores:
            return np.nan

        suma_pesos = sum(peso for _, peso in valores)
        return sum(valor * (peso / suma_pesos) for valor, peso in valores)

    df["prestigio_cat"] = df.apply(calcular_prestigio, axis=1)
    return df 

def crear_gold(ttl: pd.DataFrame, spi: pd.DataFrame):

    df = merge_dataframes(ttl, spi)
    df.dropna(subset=['alto_mm', 'ancho_mm', 'peso', 'grueso', 'n_paginas', 'precio'], inplace=True)
    df.drop(columns=['sinopsis'], inplace=True)

    columnas = ["alto_mm", "ancho_mm", "peso"]
    mask = pd.Series(True, index=df.index)

    for col in columnas:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        limite_inferior = max(0.0, q1 - 1.5 * iqr)
        limite_superior = q3 + 1.5 * iqr

        mask &= (df[col] >= limite_inferior) & (df[col] <= limite_superior)

    df = df[mask]
    

    df = portabilidad(df)
    df = compacidad(df)
    df = prestancia(df)

    df = colaboradores(df)
    df = ap_critico(df)
    df = prestigio(df)

    return df

In [86]:
# SILVER
# ttl
print("Capa Silver. Todos tus libros")
# data = crear_df()
# silver_ttl = limpiar_df_completa(data)
# spi
# print("Capa Silver. SPI")
# sel = merge_spi()
# silver_spi = prestigio_editorial(sel)

# GOLD
print("Capa Gold")
gold_final = crear_gold(silver_ttl, silver_spi)


# para valores imputados por minmax: eliminar antes los nan
# para colaboradores practicamente nulos: +83000 son cero. ver qué hacer
# cat/subcat vacías: todas. problema heredado de ttl
# prestigios nulos: sí es normal, no tenemos editoriales puntuadas en esas categorías. 0 sí que significa no registro, de hecho, si una editorial no está en el spi debe igualmente incluirse en la tabla del spi con todos los prestigios iguales a 0
# columna a eliminar: resultado de guardar en csv, irrelevante

Capa Silver. Todos tus libros
Capa Gold


In [87]:
gold_final[['alto_mm', 'ancho_mm', 'grueso', 'peso', 'precio']].describe()


,alto_mm,ancho_mm,grueso,peso,precio
count,64603.000000,64603.000000,64603.000000,64603.000000,64603.000000
mean,202.135574,134.714920,16.485758,369.873867,14.621439
std,23.087887,16.155843,12.573239,173.764326,9.266585
min,120.000000,90.000000,0.040000,2.000000,0.010000
25%,190.000000,125.000000,9.760000,240.000000,8.950000
50%,200.000000,131.000000,14.560000,343.000000,13.460000
75%,220.000000,148.000000,21.440000,468.928000,18.950000
max,280.000000,186.000000,900.000000,903.360000,288.460000


In [88]:
gold_final.to_csv("prueba_gold_2.csv")

In [109]:
print(sorted(set(silver_spi['editorial'])))
print(sorted(set(gold_final['editorial'])))
print(sorted(EDITORIALES.keys()))

['Acantilado', 'Alfaguara', 'Alianza Editorial', 'Anagrama', 'Booket', 'Castalia Ediciones', 'Debolsillo', 'Destino', 'Ediciones Akal', 'Ediciones Cátedra', 'Espasa', 'Gredos', 'Plaza & Janés', 'RAE', 'Seix Barral', 'Siruela', 'Tusquets']
['Acantilado', 'Alfaguara', 'Alianza Editorial', 'Anagrama', 'Booket', 'Castalia Ediciones', 'Debolsillo', 'Destino', 'Ediciones Akal', 'Ediciones Cátedra', 'Espasa', 'Gredos', 'Planeta', 'RAE', 'Seix Barral', 'Siruela', 'Tusquets']
['Acantilado', 'Alfaguara', 'Alianza Editorial', 'Anagrama', 'Booket', 'Castalia Ediciones', 'Debolsillo', 'Destino', 'Ediciones Akal', 'Ediciones Cátedra', 'Espasa', 'Gredos', 'Planeta', 'Plaza & Janés', 'RAE', 'Seix Barral', 'Siruela', 'Tusquets']


In [100]:
print(len(EDITORIALES))

editoriales_finales = set(gold_final['editorial'])
print(len(editoriales_finales))

18
17


In [11]:
final_ttl.columns

Index(['ean', 'titulo', 'editorial', 'coleccion', 'autoria',
       'traductor_y_editor', 'otros_contribuidores', 'aparato_critico',
       'categorias', 'subcategorias', 'idioma_original',
       'idioma_de_publicacion', 'fecha_publicacion', 'n_paginas', 'precio',
       'alto_mm', 'ancho_mm', 'grueso', 'peso', 'encuadernacion', 'es_escolar',
       'es_ibd', 'sinopsis', 'url', 'img'],
      dtype='str')